# Multi-Dataset Benchmark Verification: HGWO-PSO for BMS Safety Optimization
**Autonomous Electric Vehicle Battery Management System (BMS)**
Co-Optimization of Physical Sensor Selection & Classifier Hyperparameters for Critical Fault Detection

This notebook verifies the final experimental benchmark values reported in:
- **Table 7.1**: EV Fast-Charging Thermal Runaway Dataset (1,000 samples, 17 sensors)
- **Table 7.2**: NASA Li-ion Battery Classic 4-Cell Benchmark (636 cycles, 15 sensors)
- **Section 7.3.2**: NASA Global Multi-Temperature Archive (2,750 cycles across 34 cells, 4°C-44°C)

Strict 5-Fold Stratified Cross-Validation on physical safety failure targets.

In [ ]:
# 1. Clone repository (if running in Google Colab)
import os
if not os.path.exists("EV_Battery_Charging_TR_Dataset_with_Notes.csv"):
    !git clone https://github.com/abdullahgni/BMS.git
    %cd BMS

# Verify required packages
!pip install -q pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import warnings
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

def calc_critical_mdr(y_true, y_pred, crit_class=0):
    true_crit = (y_true == crit_class)
    pred_non_crit = (y_pred != crit_class)
    missed = np.sum(true_crit & pred_non_crit)
    total_crit = np.sum(true_crit)
    return float(missed / total_crit) if total_crit > 0 else 0.0

def evaluate_cv(X_subset, y, clf_builder, n_splits=5, seed=42, crit_class=0):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    accs, f1s, mdrs = [], [], []
    for tr, te in skf.split(X_subset, y):
        clf = clf_builder()
        clf.fit(X_subset.iloc[tr], y[tr])
        preds = clf.predict(X_subset.iloc[te])
        accs.append(accuracy_score(y[te], preds))
        f1s.append(f1_score(y[te], preds, average="macro"))
        mdrs.append(calc_critical_mdr(y[te], preds, crit_class=crit_class))
    return np.mean(accs), np.mean(f1s), np.mean(mdrs)

In [ ]:
# ==============================================================================
# DATASET 1: EV Fast-Charging Thermal Runaway Benchmark (1,000 samples)
# ==============================================================================
df_ev = pd.read_csv("EV_Battery_Charging_TR_Dataset_with_Notes.csv")
le = LabelEncoder()
df_ev["ChargingStage"] = le.fit_transform(df_ev["ChargingStage"])
df_ev["MoistureDetected"] = df_ev["MoistureDetected"].astype(int)
y_ev = df_ev["EventFlag"].map({"Runaway": 0, "Watch": 1, "Alarm": 2, "Warning": 2}).values

drop_cols = ["Timestamp", "ChargerID", "CellID", "Notes", "BMS_Status", "EventFlag", "TR_Probability"]
X_ev = df_ev.drop(columns=drop_cols)
feat_ev = X_ev.columns.tolist()
scaler_ev = StandardScaler()
X_ev_scaled = pd.DataFrame(scaler_ev.fit_transform(X_ev), columns=feat_ev)

ev_pipes = [
    ("Baseline 1 (BPSO - Paper)", "KNN (k=5)", ["ChargingStage", "DemandCurrent_A", "AvgTemp_C", "AmbientTemp_C", "StateOfHealth_%", "VibrationLevel_mg", "MoistureDetected", "ChargePower_kW"], lambda: KNeighborsClassifier(n_neighbors=5)),
    ("Baseline 2 (BWOA - Paper)", "KNN (k=5)", ["DemandCurrent_A", "StateOfHealth_%", "VibrationLevel_mg", "MoistureDetected"], lambda: KNeighborsClassifier(n_neighbors=5)),
    ("Baseline 3 (BHPWOA - Paper)", "KNN (k=5)", ["ChargingStage", "DemandCurrent_A", "AvgTemp_C", "AmbientTemp_C", "StateOfHealth_%", "VibrationLevel_mg", "MoistureDetected", "ChargePower_kW"], lambda: KNeighborsClassifier(n_neighbors=5)),
    ("Phase 1 (Co-Opt RF)", "RF (Co-Tuned)", ["ChargingStage", "ChargeCurrent_A", "SOC_%", "StateOfHealth_%", "MoistureDetected", "Pressure_kPa"], lambda: RandomForestClassifier(n_estimators=100, max_depth=8, class_weight={0: 2.0, 1: 1.0, 2: 1.0}, random_state=42)),
    ("Proposed HGWO-PSO (Ours)", "RF (HGWO-PSO Co-Tuned)", ["DemandVoltage_V", "SOC_%", "MaxTemp_C", "AvgTemp_C", "MoistureDetected"], lambda: RandomForestClassifier(n_estimators=100, max_depth=7, class_weight={0: 2.0, 1: 1.0, 2: 1.0}, random_state=42))
]

res_ev = []
for name, clf_desc, sensors, builder in ev_pipes:
    acc, f1, mdr = evaluate_cv(X_ev_scaled[sensors], y_ev, builder)
    res_ev.append({"Method": name, "Classifier": clf_desc, "Accuracy": f"{acc*100:.2f}%", "Macro-F1": f"{f1:.4f}", "Critical MDR": f"{mdr*100:.2f}%", "Sensors": f"{len(sensors)} / {len(feat_ev)}", "Selected Channels": ", ".join(sensors)})

print("
--- TABLE 7.1: EV THERMAL RUNAWAY BENCHMARK (5-FOLD CV) ---")
pd.DataFrame(res_ev)

In [ ]:
# ==============================================================================
# DATASET 2: NASA Li-ion Classic 4-Cell Benchmark (636 cycles)
# ==============================================================================
df_n4 = pd.read_csv("nasa_battery_benchmark_classic4.csv")
y_n4 = df_n4["Target_Label"].values
non_feats = ["battery_id", "cycle_file", "Capacity_Ah", "Health_State", "Target_Label"]
feat_n4 = [c for c in df_n4.columns if c not in non_feats]
X_n4 = df_n4[feat_n4].copy()
scaler_n4 = StandardScaler()
X_n4_scaled = pd.DataFrame(scaler_n4.fit_transform(X_n4), columns=feat_n4)

n4_pipes = [
    ("Baseline 1: BPSO (Paper)", "KNN (k=5)", ["Temp_Rise_C", "Final_Voltage_V", "Avg_Current_A", "Discharge_Duration_s"], lambda: KNeighborsClassifier(n_neighbors=5)),
    ("Baseline 2: BWOA (Paper)", "KNN (k=5)", ["Max_Temp_C", "Avg_Voltage_V", "Energy_Delivered_Wh"], lambda: KNeighborsClassifier(n_neighbors=5)),
    ("Standard Full-Feature RF", "Random Forest (Default)", feat_n4, lambda: RandomForestClassifier(n_estimators=100, random_state=42)),
    ("Proposed HGWO-PSO (Ours)", "RF (n=120, d=10, w0=3.4)", ["Discharge_Duration_s", "Energy_Delivered_Wh", "Max_Load_Current_A"], lambda: RandomForestClassifier(n_estimators=120, max_depth=10, class_weight={0: 3.4, 1: 1.0, 2: 1.0}, random_state=42))
]

res_n4 = []
for name, clf_desc, sensors, builder in n4_pipes:
    acc, f1, mdr = evaluate_cv(X_n4_scaled[sensors], y_n4, builder)
    res_n4.append({"Method": name, "Classifier": clf_desc, "Accuracy": f"{acc*100:.2f}%", "Macro-F1": f"{f1:.4f}", "Critical MDR": f"{mdr*100:.2f}%", "Sensors": f"{len(sensors)} / {len(feat_n4)}", "Selected Channels": ", ".join(sensors) if len(sensors) < 5 else "All 15 Sensors"})

print("
--- TABLE 7.2: NASA CLASSIC 4-CELL BENCHMARK (5-FOLD CV) ---")
pd.DataFrame(res_n4)

In [ ]:
# ==============================================================================
# DATASET 3: NASA Global Archive Benchmark (34 Cells, 2,750 cycles, 4°C-44°C)
# ==============================================================================
df_ng = pd.read_csv("nasa_battery_benchmark_global_2750.csv")
y_ng = df_ng["Target_Label"].values
feat_ng = [c for c in df_ng.columns if c not in non_feats]
X_ng = df_ng[feat_ng].copy()
scaler_ng = StandardScaler()
X_ng_scaled = pd.DataFrame(scaler_ng.fit_transform(X_ng), columns=feat_ng)

ng_pipes = [
    ("Baseline 1: BPSO (Paper)", "KNN (k=5)", ["Avg_Temp_C", "Avg_Current_A", "Energy_Delivered_Wh", "Max_Load_Current_A"], lambda: KNeighborsClassifier(n_neighbors=5)),
    ("Baseline 2: BWOA (Paper)", "KNN (k=5)", ["Discharge_Duration_s", "Energy_Delivered_Wh"], lambda: KNeighborsClassifier(n_neighbors=5)),
    ("Standard Full-Feature RF", "Random Forest (Default)", feat_ng, lambda: RandomForestClassifier(n_estimators=100, random_state=42)),
    ("Proposed HGWO-PSO (Ours)", "RF (n=120, d=10, w0=1.5)", ["Discharge_Duration_s", "Energy_Delivered_Wh", "Avg_Load_Voltage_V"], lambda: RandomForestClassifier(n_estimators=120, max_depth=10, class_weight={0: 1.5, 1: 1.0, 2: 1.0}, random_state=42))
]

res_ng = []
for name, clf_desc, sensors, builder in ng_pipes:
    acc, f1, mdr = evaluate_cv(X_ng_scaled[sensors], y_ng, builder)
    res_ng.append({"Method": name, "Classifier": clf_desc, "Accuracy": f"{acc*100:.2f}%", "Macro-F1": f"{f1:.4f}", "Critical MDR": f"{mdr*100:.2f}%", "Sensors": f"{len(sensors)} / {len(feat_ng)}", "Selected Channels": ", ".join(sensors) if len(sensors) < 5 else "All 15 Sensors"})

print("
--- SECTION 7.3.2: NASA GLOBAL MULTI-TEMPERATURE BENCHMARK (5-FOLD CV) ---")
pd.DataFrame(res_ng)

In [ ]:
# Multi-Dataset Visual Verification Bar Chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: EV TR MDR
methods_ev = ["BPSO", "BWOA", "BHPWOA", "Phase 1 RF", "HGWO-PSO"]
mdrs_ev = [59.14, 73.01, 59.14, 34.22, 5.89]
colors_ev = ["#d9534f", "#d9534f", "#d9534f", "#f0ad4e", "#2b8a3e"]
bars1 = axes[0].bar(methods_ev, mdrs_ev, color=colors_ev, edgecolor="black", linewidth=1.2)
axes[0].set_title("EV Thermal Runaway: Critical MDR (%)
(Lower is Safer)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Missed Detection Rate (%)", fontsize=11)
axes[0].grid(axis="y", linestyle="--", alpha=0.5)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5, f"{bar.get_height():.1f}%", ha="center", fontweight="bold")

# Panel 2: NASA Classic 4 Accuracy
methods_n4 = ["BPSO", "BWOA", "Full RF", "HGWO-PSO"]
acc_n4 = [97.48, 97.64, 99.21, 99.21]
colors_n4 = ["#6c757d", "#6c757d", "#495057", "#1864ab"]
bars2 = axes[1].bar(methods_n4, acc_n4, color=colors_n4, edgecolor="black", linewidth=1.2)
axes[1].set_title("NASA Classic 4: Accuracy (%)
(Higher is Better)", fontsize=12, fontweight="bold")
axes[1].set_ylim(95, 100.5)
axes[1].set_ylabel("Classification Accuracy (%)", fontsize=11)
axes[1].grid(axis="y", linestyle="--", alpha=0.5)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f"{bar.get_height():.2f}%", ha="center", fontweight="bold")

# Panel 3: Hardware Sensor Footprint Compression
datasets = ["EV Fast-Charging", "NASA Classic 4", "NASA Global"]
orig_sensors = [17, 15, 15]
opt_sensors = [5, 3, 3]
x = np.arange(len(datasets))
w = 0.35
axes[2].bar(x - w/2, orig_sensors, width=w, label="Full Sensor Suite", color="#adb5bd", edgecolor="black")
axes[2].bar(x + w/2, opt_sensors, width=w, label="HGWO-PSO Selected", color="#2b8a3e", edgecolor="black")
axes[2].set_title("Active Sensor Footprint Compression
(Hardware Channels)", fontsize=12, fontweight="bold")
axes[2].set_xticks(x)
axes[2].set_xticklabels(datasets, fontsize=10, fontweight="bold")
axes[2].set_ylabel("Number of Physical Sensors", fontsize=11)
axes[2].legend()
axes[2].grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()